# Data Challenge: PISA 2025 en Argentina

*Introducción al Análisis de Datos, UTN-FRH. Trabajo individual.*

## 1. Introducción

### El dataset
PISA (Programa para la Evaluación Internacional de Estudiantes) es la evaluación que organiza la OCDE cada tres años para medir qué saben hacer los estudiantes de 15 años en Matemática, Lectura y Ciencias. En la edición 2025 el dominio principal fue Ciencias. Además de la prueba, cada estudiante responde un cuestionario sobre su hogar, su escuela, sus clases y su uso de tecnología, y el director de cada escuela completa otro cuestionario sobre la institución.

Trabajo con los archivos públicos de PISA 2025 (Public Use Files), filtrados para Argentina: **11.092 estudiantes de 412 escuelas**.

- Fuente: OCDE, *PISA 2025 Database*, https://www.oecd.org/en/data/datasets/pisa-2025-database.html
- Documentación consultada: el codebook 2025 (`PISA2025_Codebook.xlsx`) y los cuestionarios de estudiantes y escuelas.

Los archivos originales están en formato SPSS (`.sav`) y el de estudiantes pesa más de 2 GB, con datos de todos los países. Por eso escribí un script aparte, `preparar_datos.py`, que lee ese archivo en pedazos, se queda con Argentina y con las variables que uso, y guarda dos archivos livianos: `pisa2025_arg_estudiantes.csv` y `pisa2025_arg_escuelas.csv`. Este notebook trabaja solo con esos dos CSV.

### Contexto y motivación
En las escuelas se repite una preocupación: estudiantes distraídos por el celular, muchas horas frente a pantallas fuera del horario escolar y, desde hace poco, el uso de chatbots de inteligencia artificial para hacer las tareas. El cuestionario de PISA 2025 incluye preguntas sobre los tres temas, entre ellas una que consulta directamente con qué frecuencia los estudiantes se distraen con dispositivos digitales en las clases de ciencias.

Este análisis se vincula con MaestrIA, un asistente de IA pensado para ayudar a los docentes a diseñar y corregir evaluaciones escritas y a dar devoluciones, siempre con el docente revisando y decidiendo. Saber qué factores acompañan a un mejor o peor desempeño ayuda a pensar en qué parte del proceso puede servir una herramienta así y en qué parte no.

### Preguntas guía
1. **¿Qué factores del entorno de aprendizaje se asocian con mayores diferencias de desempeño en Matemática, Lectura y Ciencias entre los estudiantes argentinos de 15 años: la distracción por dispositivos en clase, el tiempo de pantallas por ocio, el clima disciplinario o el nivel socioeconómico?**
2. **¿Con qué frecuencia usan los estudiantes los chatbots de IA para la escuela (resumir, investigar, redactar, aprender) y cómo se relaciona cada uso con su desempeño?**

### Alcance
- La población son estudiantes de 15 años escolarizados. Las conclusiones valen para ese grupo, no para todo el sistema educativo.
- Los datos son de un solo momento y surgen de una encuesta, así que permiten describir **asociaciones**, no establecer causas. Si los estudiantes que se distraen más rinden menos, eso no prueba que la distracción sea la causa.
- Para los puntajes uso el primer valor plausible de cada materia (`PV1MATH`, `PV1READ`, `PV1SCIE`) y no aplico los pesos muestrales de PISA. En las conclusiones detallo qué implica esta simplificación.

## 2. Exploración inicial

Primero se importan las librerías y se cargan los dos CSV preparados a partir de los archivos públicos de PISA 2025.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo de gráficos usado en la cursada
plt.style.use('seaborn-v0_8')
sns.set_palette('muted')

Si el notebook se ejecuta en Google Colab, hay que subir antes los dos CSV. La celda siguiente tiene esa parte comentada, para que funcione igual en una computadora local.

In [2]:
# Solo en Google Colab: descomentar estas líneas y elegir los dos CSV
# from google.colab import files
# files.upload()
# carpeta = ''

# En la computadora local los CSV están en la carpeta data/
carpeta = 'data/'

estudiantes = pd.read_csv(carpeta + 'pisa2025_arg_estudiantes.csv')
escuelas = pd.read_csv(carpeta + 'pisa2025_arg_escuelas.csv')

### 2.1 Dimensiones y primeras filas

In [3]:
print('Estudiantes:', estudiantes.shape)
print('Escuelas:', escuelas.shape)

Estudiantes: (11092, 55)
Escuelas: (412, 5)


In [4]:
estudiantes.head()

,CNT,CNTSCHID,CNTSTUID,MALE,ST250Q04JA,ST097Q01TA,ST097Q02TA,ST097Q03TA,ST097Q04TA,ST097Q05TA,...,PV1READ,PV2READ,PV3READ,PV4READ,PV5READ,PV6READ,PV7READ,PV8READ,PV9READ,PV10READ
0,ARG,3200001,3200001,1.0,1.0,3.0,3.0,3.0,4.0,2.0,...,510.650824,458.004549,492.843520,501.655085,510.901170,398.557368,524.778848,502.435195,458.612719,460.692049
1,ARG,3200001,3200002,0.0,1.0,3.0,2.0,3.0,2.0,NaN,...,326.301510,225.437435,370.384461,308.859792,275.441596,392.713512,353.080611,382.271740,361.252432,275.626428
2,ARG,3200001,3200003,0.0,1.0,3.0,2.0,3.0,3.0,3.0,...,513.795472,466.127827,446.296314,488.033277,477.265384,442.493769,478.840866,463.107629,494.160499,495.065453
3,ARG,3200001,3200004,0.0,1.0,2.0,3.0,1.0,3.0,1.0,...,214.364471,285.993741,273.243868,237.544934,260.356522,241.737797,312.684701,342.132885,335.226210,366.802136
4,ARG,3200001,3200005,1.0,1.0,2.0,2.0,2.0,3.0,3.0,...,381.937331,378.959362,403.866066,341.232403,348.349695,357.292023,351.330295,361.739706,336.601204,370.851343


La tabla de estudiantes tiene 11.092 filas, una por estudiante, y 55 columnas. La de escuelas tiene 412 filas, una por escuela, y se vincula con la anterior por el identificador `CNTSCHID`. En promedio hay unos 27 estudiantes por escuela.

In [5]:
estudiantes.info()

<class 'pandas.DataFrame'>
RangeIndex: 11092 entries, 0 to 11091
Data columns (total 55 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CNT         11092 non-null  str    
 1   CNTSCHID    11092 non-null  int64  
 2   CNTSTUID    11092 non-null  int64  
 3   MALE        11092 non-null  float64
 4   ST250Q04JA  10055 non-null  float64
 5   ST097Q01TA  9749 non-null   float64
 6   ST097Q02TA  9599 non-null   float64
 7   ST097Q03TA  9572 non-null   float64
 8   ST097Q04TA  9456 non-null   float64
 9   ST097Q05TA  9433 non-null   float64
 10  ST097Q06DA  9437 non-null   float64
 11  ST326Q04JA  7650 non-null   float64
 12  ST326Q05JA  7647 non-null   float64
 13  ST326Q06JA  7639 non-null   float64
 14  ST438Q01DA  7856 non-null   float64
 15  ST438Q02DA  7766 non-null   float64
 16  ST438Q03DA  7680 non-null   float64
 17  ST438Q04DA  7652 non-null   float64
 18  ST322Q06JA  7413 non-null   float64
 19  IC170Q10DA  5633 non-null   float64


Casi todas las columnas son `float64`, incluso las preguntas del cuestionario, que en realidad son categorías (1, 2, 3...). Pandas las guarda como decimales porque tienen valores vacíos. Los identificadores (`CNTSCHID`, `CNTSTUID`) son enteros y `CNT` es texto.

La cantidad de valores no nulos varía mucho entre columnas. Los puntajes están completos (11.092), mientras que las preguntas del cuestionario tienen entre 5.600 y 10.200 respuestas. Eso se analiza en la sección de limpieza.

In [6]:
escuelas.info()

<class 'pandas.DataFrame'>
RangeIndex: 412 entries, 0 to 411
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CNT         412 non-null    str    
 1   CNTSCHID    412 non-null    int64  
 2   SC001Q01TA  352 non-null    float64
 3   SC190Q02JA  349 non-null    float64
 4   PRIVATESCH  412 non-null    float64
dtypes: float64(3), int64(1), str(1)
memory usage: 16.2 KB


En escuelas, el tipo de gestión (`PRIVATESCH`, que después llamo `gestion`) está completo porque surge del diseño de la muestra. El tamaño de la localidad y la prohibición de celulares vienen del cuestionario del director y tienen unas 60 escuelas sin respuesta.

### 2.2 Qué significa cada variable
Los nombres originales de las columnas son códigos de PISA (por ejemplo, `ST250Q04JA`), que no dicen nada por sí solos. La tabla resume las variables que uso, con su código original, el nombre más descriptivo que les doy en la sección 2.3, su significado y su escala. Los significados y escalas salen del codebook 2025.

| Código PISA | Nombre en el notebook | Significado | Escala |
|---|---|---|---|
| `CNT` | `pais` | País (todas las filas son ARG) | texto |
| `CNTSCHID`, `CNTSTUID` | `id_escuela`, `id_estudiante` | Identificador de escuela y de estudiante | número |
| `MALE` | `genero` | Género según los datos de muestreo | 1 = varón, 0 = mujer |
| `REPEAT` | `repitio` | Repitió al menos un año | 1 = sí, 0 = no |
| `PV1MATH` … `PV10MATH` | `pv1_matematica` … `pv10_matematica` | Valores plausibles del puntaje en Matemática (igual para Lectura y Ciencias: `pv1_lectura`, `pv1_ciencias`...) | escala PISA |
| `ESCS` | `indice_socioeconomico` | Índice de nivel económico, social y cultural del hogar | continuo; 0 = promedio OCDE |
| `ST097Q01TA` | `clima_no_escuchan` | En clase de ciencias, los estudiantes no escuchan al docente | 1 = todas las clases … 4 = nunca o casi nunca |
| `ST097Q02TA` | `clima_ruido` | Hay ruido y desorden | ídem |
| `ST097Q03TA` | `clima_esperan_silencio` | El docente tiene que esperar mucho a que se calmen | ídem |
| `ST097Q04TA` | `clima_no_trabajan` | Los estudiantes no pueden trabajar bien | ídem |
| `ST097Q05TA` | `clima_tardan_empezar` | Tardan mucho en empezar a trabajar | ídem |
| `ST097Q06DA` | `distraccion_dispositivos` | Los estudiantes se distraen usando dispositivos digitales (celulares, sitios, apps) | ídem |
| `DISCLISCI` | `indice_clima_disciplinario` | Índice de clima disciplinario en ciencias (más alto = mejor clima) | continuo |
| `ST326Q04JA` | `ocio_horas_escuela` | Horas diarias de uso de dispositivos por ocio en la escuela | 1 = nada, 2 = hasta 1 h … 9 = más de 7 h |
| `ST326Q05JA` | `ocio_horas_dias_habiles` | Ídem, antes y después de la escuela | ídem |
| `ST326Q06JA` | `ocio_horas_finde` | Ídem, los fines de semana | ídem |
| `IC177Q02JA` | `redes_horas_dia_habil` | Tiempo en redes sociales en un día de semana | 1 = nada, 2 = menos de 1 h, 3 = 1 a 3 h, 4 = 3 a 5 h, 5 = 5 a 7 h, 6 = más de 7 h |
| `ST322Q06JA` | `presion_responder_en_clase` | Se siente presionado a estar en línea y responder mensajes durante la clase | 1 = nunca … 5 = siempre; 6 = no aplica |
| `ST250Q04JA` | `celular_propio` | Tiene celular propio con internet | 1 = sí, 2 = no |
| `ST438Q01DA` | `ia_resumir` | Usa chatbots de IA para la escuela para resumir un texto | 1 = nunca … 5 = todos los días |
| `ST438Q02DA` | `ia_investigar` | … para investigar un tema nuevo | ídem |
| `ST438Q03DA` | `ia_redactar` | … para redactar trabajos escritos | ídem |
| `ST438Q04DA` | `ia_aprender` | … para ayudarse a aprender | ídem |
| `AIUSESCH` | `indice_uso_ia` | Índice de uso de IA en la escuela | continuo |
| `IC170Q10DA` | `ia_frecuencia_escuela` | Frecuencia de uso de herramientas de IA en la escuela | 1 = nunca … 5 = varias veces por día; 6 = no disponible |
| `PRIVATESCH` (escuelas) | `gestion` | Gestión de la escuela | 1 = pública, 2 = privada |
| `SC001Q01TA` (escuelas) | `tamanio_localidad` | Tamaño de la localidad | 1 = rural (menos de 3.000 hab.) … 6 = más de 10 millones |
| `SC190Q02JA` (escuelas) | `prohibe_celulares` | No se permite el uso de celulares en la escuela | 1 = sí, 2 = no |

Las preguntas del cuestionario son categorías guardadas como números. Hay que tener cuidado con el orden: en las preguntas de clima y distracción, un número más bajo indica una situación **más frecuente** (1 = en todas las clases).

### 2.3 Nombres más descriptivos
Renombro las columnas con `.rename()` usando un diccionario. Para los 30 valores plausibles armo los nombres con un ciclo `for`, en lugar de escribirlos uno por uno. Los CSV conservan los códigos originales, así que cualquier columna se puede rastrear en el codebook con la tabla anterior.

In [7]:
nombres_estudiantes = {
    'CNT': 'pais',
    'CNTSCHID': 'id_escuela',
    'CNTSTUID': 'id_estudiante',
    'MALE': 'genero',
    'REPEAT': 'repitio',
    'ESCS': 'indice_socioeconomico',
    'ST097Q01TA': 'clima_no_escuchan',
    'ST097Q02TA': 'clima_ruido',
    'ST097Q03TA': 'clima_esperan_silencio',
    'ST097Q04TA': 'clima_no_trabajan',
    'ST097Q05TA': 'clima_tardan_empezar',
    'ST097Q06DA': 'distraccion_dispositivos',
    'DISCLISCI': 'indice_clima_disciplinario',
    'ST326Q04JA': 'ocio_horas_escuela',
    'ST326Q05JA': 'ocio_horas_dias_habiles',
    'ST326Q06JA': 'ocio_horas_finde',
    'IC177Q02JA': 'redes_horas_dia_habil',
    'ST322Q06JA': 'presion_responder_en_clase',
    'ST250Q04JA': 'celular_propio',
    'ST438Q01DA': 'ia_resumir',
    'ST438Q02DA': 'ia_investigar',
    'ST438Q03DA': 'ia_redactar',
    'ST438Q04DA': 'ia_aprender',
    'AIUSESCH': 'indice_uso_ia',
    'IC170Q10DA': 'ia_frecuencia_escuela',
}

# Valores plausibles: PV1MATH -> pv1_matematica, PV2MATH -> pv2_matematica, etc.
materias = {'MATH': 'matematica', 'READ': 'lectura', 'SCIE': 'ciencias'}
for codigo, materia in materias.items():
    for i in range(1, 11):
        nombres_estudiantes[f'PV{i}{codigo}'] = f'pv{i}_{materia}'

estudiantes = estudiantes.rename(columns=nombres_estudiantes)

escuelas = escuelas.rename(columns={
    'CNT': 'pais',
    'CNTSCHID': 'id_escuela',
    'PRIVATESCH': 'gestion',
    'SC001Q01TA': 'tamanio_localidad',
    'SC190Q02JA': 'prohibe_celulares',
})

print(estudiantes.columns.tolist())
print(escuelas.columns.tolist())

['pais', 'id_escuela', 'id_estudiante', 'genero', 'celular_propio', 'clima_no_escuchan', 'clima_ruido', 'clima_esperan_silencio', 'clima_no_trabajan', 'clima_tardan_empezar', 'distraccion_dispositivos', 'ocio_horas_escuela', 'ocio_horas_dias_habiles', 'ocio_horas_finde', 'ia_resumir', 'ia_investigar', 'ia_redactar', 'ia_aprender', 'presion_responder_en_clase', 'ia_frecuencia_escuela', 'redes_horas_dia_habil', 'indice_clima_disciplinario', 'indice_uso_ia', 'indice_socioeconomico', 'repitio', 'pv1_ciencias', 'pv2_ciencias', 'pv3_ciencias', 'pv4_ciencias', 'pv5_ciencias', 'pv6_ciencias', 'pv7_ciencias', 'pv8_ciencias', 'pv9_ciencias', 'pv10_ciencias', 'pv1_matematica', 'pv2_matematica', 'pv3_matematica', 'pv4_matematica', 'pv5_matematica', 'pv6_matematica', 'pv7_matematica', 'pv8_matematica', 'pv9_matematica', 'pv10_matematica', 'pv1_lectura', 'pv2_lectura', 'pv3_lectura', 'pv4_lectura', 'pv5_lectura', 'pv6_lectura', 'pv7_lectura', 'pv8_lectura', 'pv9_lectura', 'pv10_lectura']
['pais', 'i

### 2.4 Estadísticas descriptivas
Primero, los puntajes (primer valor plausible de cada materia) y los tres índices continuos.

In [8]:
columnas_numericas = ['pv1_matematica', 'pv1_lectura', 'pv1_ciencias',
                      'indice_socioeconomico', 'indice_clima_disciplinario', 'indice_uso_ia']
estudiantes[columnas_numericas].describe().round(2)

,pv1_matematica,pv1_lectura,pv1_ciencias,indice_socioeconomico,indice_clima_disciplinario,indice_uso_ia
count,11092.00,11092.00,11092.00,9808.00,9616.00,7728.00
mean,383.63,402.70,413.38,-0.50,-0.38,0.26
std,81.39,92.04,92.57,1.05,0.98,1.01
min,118.61,77.51,114.31,-4.09,-2.62,-1.58
25%,325.27,336.33,344.91,-1.29,-0.99,-0.34
50%,378.98,400.14,409.90,-0.55,-0.41,0.14
75%,438.35,466.83,478.83,0.35,0.26,0.84
max,707.25,741.06,767.41,3.26,1.99,2.34


Los promedios son 383,6 puntos en Matemática, 402,7 en Lectura y 413,4 en Ciencias. Como referencia, la escala de PISA se construyó para que el promedio de los países de la OCDE rondara los 500 puntos. La dispersión es grande: en Matemática, el 25 % de los estudiantes con menor puntaje queda por debajo de 325 puntos y el 25 % con mayor puntaje supera los 438.

El índice socioeconómico (`indice_socioeconomico`) tiene una media de −0,50: en promedio, los hogares de estos estudiantes están medio desvío por debajo del promedio de la OCDE. El índice de clima disciplinario también es negativo en promedio (−0,38).

### 2.5 Primeras frecuencias
Con qué frecuencia los estudiantes dicen que en sus clases de ciencias hay distracción por dispositivos (porcentaje de los que respondieron):

In [9]:
# 1 = en todas las clases, 2 = en la mayoría, 3 = en algunas, 4 = nunca o casi nunca
estudiantes['distraccion_dispositivos'].value_counts(normalize=True).sort_index().mul(100).round(1)

distraccion_dispositivos
1.0    28.6
2.0    24.6
3.0    27.3
4.0    19.5
Name: proportion, dtype: float64

El 28,6 % dice que hay distracción con dispositivos en todas las clases de ciencias y el 24,6 % en la mayoría. Es decir, más de la mitad la percibe en casi todas las clases. Solo el 19,5 % responde que nunca o casi nunca pasa.

In [10]:
# Uso de chatbots de IA 'para ayudarme a aprender'
# 1 = nunca, 2 = 1 o 2 veces al año, 3 = 1 o 2 veces al mes, 4 = 1 o 2 veces por semana, 5 = todos los días
estudiantes['ia_aprender'].value_counts(normalize=True).sort_index().mul(100).round(1)

ia_aprender
1.0    15.4
2.0    13.5
3.0    18.6
4.0    25.8
5.0    26.8
Name: proportion, dtype: float64

El uso de IA está muy extendido: el 52,6 % de los que respondieron usa chatbots para ayudarse a aprender al menos una vez por semana, y solo el 15,4 % dice que nunca lo hace.

In [11]:
print(escuelas['gestion'].value_counts())
print(escuelas['prohibe_celulares'].value_counts(dropna=False))

gestion
1.0    277
2.0    135
Name: count, dtype: int64
prohibe_celulares
2.0    222
1.0    127
NaN     63
Name: count, dtype: int64


De las 412 escuelas, 277 son de gestión pública y 135 de gestión privada. Entre las que respondieron, 127 no permiten el uso de celulares y 222 sí lo permiten; 63 no informaron.

## 3. Limpieza de datos

*(Pendiente.)*

## 4. Transformaciones

*(Pendiente.)*

## 5. Visualización

*(Pendiente.)*

## 6. Conclusiones

*(Pendiente: respuesta a las preguntas guía, reflexión y limitaciones.)*